# Assignment 2B — RAG Pipeline
*Chunking, dense / keyword / hybrid search, reranking, grounded generation and an LLM judge*

**Domain:** Medical and clinical literature (Type 2 Diabetes management)

**Generator:** Assignment 1 model — `mistralai/Mistral-7B-v0.1` + CPT checkpoint + QLoRA adapter (Assignment 1 model)

| Part | What we do | Marks |
| --- | --- | --- |
| A | Dense search and chunk size | 5 |
| B | Dense vs BM25 vs hybrid (RRF) | 4 |
| C | Cross-encoder reranking | 4 |
| D | Grounded answers and an LLM judge | 7 |
| **Total** | | **20** |

**Our choice:** the final run is on the lab server, where the Assignment 1 model files are. The brief asks for one notebook for the whole assignment, with all outputs visible.

**Models used, and where each choice comes from**

| Role | Model | Source |
| --- | --- | --- |
| Generator | `mistralai/Mistral-7B-v0.1` + CPT + QLoRA adapter (Assignment 1 model) | PDF Overview (p1): "your fine-tuned model as the answer generator" |
| Embedding | `BAAI/bge-base-en-v1.5` | PDF A0 (p1): "for example 256 tokens for all-MiniLM-L6-v2". PDF A2 (p1): "either switch to a model with a longer limit or replace that size" (we chose a model with a longer limit, as the rule allows) |
| Vector index | FAISS `IndexFlatIP` | PDF A1.1 (p1): "builds a FAISS IndexFlatIP index (or another vector index)" |
| Keyword search | BM25 with `rank-bm25` | PDF B2 (p2): "build a BM25 index". The PDF names no library; `rank-bm25` is our choice |
| Reranker | `cross-encoder/ms-marco-MiniLM-L-6-v2` | PDF C1 (p3): "for example cross-encoder/ms-marco-MiniLM-L-6-v2" |
| Judge | `Qwen/Qwen2.5-7B-Instruct` | PDF D2 (p4): "Use a judge model different from your generator, for example Llama 3.1 8B Instruct or Qwen2.5 7B Instruct, and name it" |

**Setup (Assignment 1 conventions, our choice, not in the brief).** This cell follows the Assignment 1 notebook: the same installs, the same device check, and the same Hugging Face cache. It is needed on the Kubeflow lab server.

**PDF step:** none. Setup for this notebook: it finds the storage folder, the Hugging Face cache and the GPU.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: finds the platform, picks a storage folder, sets the Hugging Face cache and checks the GPU.
# Why it runs first: Hugging Face reads HF_HOME when transformers is imported, so it must be set before that import.
# Output: the environment, device, dtype, GPU name and folders used.
# ---------------------------------------------------------------------------
import importlib.util
import os
from types import SimpleNamespace


def detect_env():
    """Return "kubeflow", "colab" or "local". Kubeflow is checked first."""
    # The lab server sets NB_PREFIX or KUBERNETES_SERVICE_HOST. Either one means Kubeflow.
    if ("NB_PREFIX" in os.environ or "KUBERNETES_SERVICE_HOST" in os.environ
            or os.path.isdir("/home/jovyan/data")):
        return "kubeflow"
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "colab"
    except ModuleNotFoundError:      # the "google" package is missing, so this is not Colab
        pass
    return "local"


def setup():
    """Pick the storage folder, set the Hugging Face cache, and read the device and dtype."""
    env = detect_env()
    # Folders to try for storage, per platform. Colab wipes /content when the session ends.
    candidates = {"colab": ["/content"], "kubeflow": ["/home/jovyan/assessment", "/home/jovyan/data"],
                  "local": [os.path.expanduser("~/lab-data"), os.getcwd()]}[env]
    root = os.getcwd()
    # Try the folders in order. The first one we can write to becomes the storage root.
    for path in candidates:
        try:
            os.makedirs(path, exist_ok=True)
            # Write and delete a small file to prove the folder is writable.
            probe = os.path.join(path, ".write_test")
            open(probe, "w").close()
            os.remove(probe)
            root = path
            break
        except Exception:
            continue

    # The model cache goes inside the storage root. This must be set before transformers is imported.
    cache_root = os.path.join(root, "hf-cache")
    os.environ["HF_HOME"] = cache_root
    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # no download bars, which would show as empty lines in the notebook

    # Device and dtype. The defaults are for a CPU-only run. A GPU replaces them below.
    device, dtype, gpu_name, gpu_mem = "cpu", "float32", None, 0.0
    try:
        import torch
        if torch.cuda.is_available():
            device = "cuda"
            gpu_name = torch.cuda.get_device_name(0)
            gpu_mem = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
            # bf16 when the GPU supports it. It has the same range as fp32 and half the memory.
            dtype = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
    except Exception:
        pass

    state = SimpleNamespace(ENV=env, ROOT=root, CACHE_ROOT=cache_root, DEVICE=device,
                            DTYPE=dtype, IS_GPU=(device == "cuda"), GPU_NAME=gpu_name, GPU_MEM_GB=gpu_mem)
    print(f"env={env}  device={device}  dtype={dtype}  gpu={gpu_name or 'none'} ({gpu_mem} GB)")
    print(f"storage root : {root}")
    print(f"HF cache     : {cache_root}")
    return state


# Run the setup once. Later cells read the results from L (for example L.DEVICE and L.DTYPE).
L = setup()


**PDF step:** none directly. It checks the inputs the brief reuses: "your cleaned domain .txt files as the document collection, 10 queries from your Assignment 1 evaluation set" (PDF Overview, p1).

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: sets the input and output paths, checks that the inputs exist, and fixes the random seed.
# Why: every later step reads these paths. A missing file should stop the notebook here, not halfway through a run.
# Output: OK or MISSING for each input.
# ---------------------------------------------------------------------------

import json
import random
import re
from pathlib import Path

import numpy as np

# Fixed seeds. Any random step gives the same result on every run.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Kubeflow lab layout. This notebook is /home/jovyan/assessment/Ass-2/Assignment2B.ipynb, so ../Ass-1 is
# /home/jovyan/assessment/Ass-1 (shown as /assessment/Ass-1 in the Jupyter file browser). The setup cell puts the
# Hugging Face cache at /home/jovyan/assessment/hf-cache.
ASS1 = Path("../Ass-1").resolve()
CORPUS_DIR = ASS1 / "domain_corpus"                      # 14 cleaned .txt files
DATASET_PATH = ASS1 / "instruction_dataset.jsonl"        # 400 rows. The LAST 80 rows are the eval set.
# Do not use ASS1 / "instruction_eval.jsonl". It has 93 pairs, none of them in instruction_dataset.jsonl.
# The Assignment 1 B1 output says the eval split has 80 pairs, which are the last 80 rows of the dataset.

# Assignment 1 saved the model files under ASS1 / "persistent" (our choice, from Assignment 1 cells 29 and 49).
# The prints below show where these paths resolve.
CPT_DIR = ASS1 / "persistent" / "cpt_ckpt"
ADAPTER_DIR = ASS1 / "persistent" / "adapter"
print(f"Assignment 1 folder : {ASS1}")
print(f"CPT checkpoint      : {CPT_DIR}")
print(f"QLoRA adapter       : {ADAPTER_DIR}")
print(f"HF cache            : {L.CACHE_ROOT}")

# Folders for this notebook's own files: data/ for inputs it writes, outputs/ for tables.
DATA_DIR, OUT_DIR = Path("data"), Path("outputs")
# Create the folders if they are missing. exist_ok=True means nothing fails if they already exist.
for folder in (DATA_DIR, OUT_DIR):
    folder.mkdir(exist_ok=True)


def count_rows(path):
    """Number of non-empty lines in a JSONL file."""
    with open(path, encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


# Each check is a label and a True or False result. Printing them shows which inputs are missing.
checks = [
    ("corpus: 14 .txt files", CORPUS_DIR.is_dir() and len(list(CORPUS_DIR.glob("*.txt"))) == 14),
    ("instruction_dataset.jsonl: 400 rows", DATASET_PATH.exists() and count_rows(DATASET_PATH) == 400),
    ("CPT checkpoint folder (lab server only)", CPT_DIR.is_dir()),
    ("QLoRA adapter folder (lab server only)", ADAPTER_DIR.is_dir()),
]
for label, ok in checks:
    print(f"{'OK' if ok else 'MISSING':8} {label}")


## Prerequisites — Environment Setup

We install the packages that are missing and print their versions, so anyone can see which versions gave these results.

**PDF step:** none. Setup: installs the Python packages this notebook imports, if they are missing.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: installs the packages that are missing.
# Why: the notebook has to run on a fresh lab server or Colab runtime.
# Output: which packages were already there and which were installed.
# ---------------------------------------------------------------------------

import importlib.util
import subprocess
import sys

# pip name -> import name. The comment after each line says which part of the brief uses it.
PACKAGES = {
    "sentence-transformers": "sentence_transformers",  # embeddings (A, B) and the CrossEncoder reranker (C)
    "faiss-cpu": "faiss",                              # IndexFlatIP vector index (A, B)
    "rank-bm25": "rank_bm25",                          # BM25Okapi keyword index (B)
    "transformers": "transformers",                    # tokenizers, generator and judge (A, D)
    "peft": "peft",                                    # QLoRA adapter (D)
    "bitsandbytes": "bitsandbytes",                    # 4-bit (nf4) loading, as Assignment 1 cell 5 (D)
    "accelerate": "accelerate",                        # device_map loading (D)
    "pandas": "pandas",                                # tables
}

# Keep only the packages whose import name cannot be found in this Python.
missing = [pip for pip, module in PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("installing:", ", ".join(missing))
    # Run this Python's own pip, so the packages go into the environment this notebook uses.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("all packages were already installed")
print("already there:", ", ".join(p for p in PACKAGES if p not in missing))


**PDF step:** none. A helper used by the tables in Parts A to D, so every table has the same layout.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines print_table(), a helper that prints aligned text tables.
# Every table in this notebook uses it. Column widths come from the data, so the columns stay aligned.
# Output: nothing (definitions only).
# ---------------------------------------------------------------------------
def print_table(headers, rows, title=None, align=None, save_as=None):
    """Print an aligned text table.

    headers   : column titles
    rows      : list of tuples. Each cell is already a string, so the caller controls
                separators and the dash used for values that do not apply.
    title     : optional heading printed above the table
    align     : per-column "l" or "r". The default is label on the left and numbers on the right.
    save_as   : optional file name. The table is also saved as outputs/<save_as>.csv.
    """
    # Default alignment: first column left, the rest right.
    align = align or ["l"] + ["r"] * (len(headers) - 1)
    # Keep the rows in a list, so they can be measured here and printed again below.
    body = list(rows)
    # Each column is as wide as its longest cell, header included.
    widths = [max(len(headers[c]), max(len(r[c]) for r in body)) for c in range(len(headers))]
    gap = "   "  # space between columns
    # Total width of the table, used for the lines drawn above and below.
    width = sum(widths) + len(gap) * (len(widths) - 1)

    # Format one row. Each cell is padded to its column width.
    def line(cells):
        """Format one table row: pad each cell to its column width."""
        return gap.join(c.ljust(w) if a == "l" else c.rjust(w)
                        for c, w, a in zip(cells, widths, align))

    # Print the title, the header and the rows.
    if title:
        print("=" * width)
        print(title)
    print("=" * width)
    print(line(headers))
    print("-" * width)
    for r in rows:
        print(line(r))
    print("=" * width)

    if save_as:
        import csv
        with open(OUT_DIR / f"{save_as}.csv", "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(headers)
            writer.writerows(rows)


def show_md_table(headers, rows, title=None, save_as=None):
    """Show a table as Markdown, so long text wraps in the notebook.

    Used for tables with long text, such as queries, spans and judge reasons.
    save_as also writes outputs/<save_as>.csv.
    """
    from IPython.display import Markdown, display

    def cell(text):
        """Keep a table cell on one line, and replace "|" so it does not start a new column."""
        return " ".join(str(text).split()).replace("|", "/")

    md = f"**{title}**\n\n" if title else ""
    md += "| " + " | ".join(headers) + " |\n| " + " | ".join("---" for _ in headers) + " |\n"
    md += "\n".join("| " + " | ".join(cell(c) for c in r) + " |" for r in rows)
    display(Markdown(md))
    if save_as:
        import csv
        with open(OUT_DIR / f"{save_as}.csv", "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(headers)
            writer.writerows(rows)


**PDF step:** none. A helper that shows a status line during slow steps, such as loading a model.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines working(), a context manager that shows a status line during slow steps.
# Why: loading the 7B models takes minutes. The status line shows the notebook is still running.
# Output: nothing (definitions only). Each slow step prints its time when it ends.
# ---------------------------------------------------------------------------
import threading
import time
from pathlib import Path


def _folder_gb(folder):
    """Size of everything inside a folder, in GB. It is 0 if the folder does not exist yet."""
    try:
        # Add up the size of each real file. Symlinks are skipped, so one file is not counted twice.
        return sum(f.stat().st_size for f in Path(folder).rglob("*") if f.is_file() and not f.is_symlink()) / 1024**3
    except OSError:
        return 0.0


class working:
    """Context manager. It shows 'label ... 2m10s' (and the folder size, if a folder is given) while the code inside runs."""

    def __init__(self, label, folder=None, expected_gb=None):
        """Keep the label, the folder to measure, and the expected final size of that folder."""
        self.label, self.folder, self.expected_gb = label, folder, expected_gb

    def _line(self):
        """Build the status text for this moment."""
        # Whole seconds since the step started, shown as minutes and seconds.
        secs = int(time.time() - self.t0)
        text = f"{self.label} ... {secs // 60}m{secs % 60:02d}s"
        if self.folder is not None:
            gb = _folder_gb(self.folder)
            text += f"  ({gb:.1f} of ~{self.expected_gb:.1f} GB)" if self.expected_gb else f"  ({gb:.1f} GB so far)"
        return text

    def _run(self):
        """Background loop. It rewrites the status line every 5 seconds until the step ends."""
        while not self._stop.wait(5):
            print("\r" + self._line() + "   ", end="", flush=True)

    def __enter__(self):
        """Start the status line in a background thread."""
        self.t0, self._stop = time.time(), threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc):
        """Stop the status line and print the final time."""
        self._stop.set()
        self._thread.join()
        # Replace ' ...' with ' ... done in' once, so the last line reads as the time the step took.
        print("\r" + self._line().replace(" ...", " ... done in", 1) + "   ")
        return False


## Before Part A — 10 queries and their answer spans

**Goal:** fix the 10 queries, their expected answers and their answer spans. Parts A, B and C score every search with these spans.

**What we do (our choice, not in the brief):**
1. Take the Assignment 1 eval set: the last 80 rows of `instruction_dataset.jsonl`. The dataset has no passage field, so each row is matched to the corpus.
2. For each row, find the corpus sentence (8 to 40 words) with the most word overlap with the expected answer. That sentence is the span, and its file is the passage.
3. Keep spans that occur exactly once in the corpus. Keep the best row per file and take 10 rows.
4. The expected answer is the row's `response`. The judge in D2 uses it.
5. Human check: read each span against its query, and replace any span that does not answer it.

A retrieved chunk is a **hit** if it contains the answer span.

*Our choice: the course may not accept this matching method, so confirm it before the final run.*


**PDF step (Before you start, p1):** "for each of your 10 queries, mark the answer span, the sentence or short passage in your corpus that answers it." The 10 queries come from the Assignment 1 evaluation set.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: loads the last 80 eval rows, matches each answer to a corpus sentence, and picks 10 queries, one per file.
# Why: the brief needs one answer span per query. This matching method is our choice (see the notes above).
# Output: the 10 queries, each with its file and its span.
# ---------------------------------------------------------------------------

# The eval set is the last 80 rows of the Assignment 1 dataset.
eval_rows = [json.loads(line) for line in open(DATASET_PATH, encoding="utf-8")][-80:]
assert len(eval_rows) == 80

# Common English words that do not help the match, so they are ignored.
STOP = set("the a an of and or to in on for with by is are was were be as at from that this it its which".split())

# The set of lowercase words (letters and digits) in a text, without the stop words.
def tokens(text):
    return set(re.findall(r"[a-z0-9]+", text.lower())) - STOP

# Candidate spans: every corpus sentence of 8 to 40 words, with the file it comes from.
SENTENCES = []
for f in sorted(CORPUS_DIR.glob("*.txt")):
    text = " ".join(f.read_text(encoding="utf-8").split())
    # Split after . ! or ? followed by a space, which gives one sentence per item.
    for sent in re.split(r"(?<=[.!?])\s+", text):
        if 8 <= len(sent.split()) <= 40:
            SENTENCES.append((f.name, sent))

def best_span(response):
    """Find the corpus sentence that shares the most words with the expected answer."""
    want = tokens(response)
    scored = []
    for source, sent in SENTENCES:
        have = tokens(sent)
        if have:
            overlap = len(want & have)
            # Score = shared words divided by the square root of the sentence's word count.
            # Long sentences do not win just by being long.
            # Tuple order for max(): score, then overlap, then -length (so a tie goes to the shorter sentence).
            scored.append((overlap / len(have) ** 0.5, overlap, -len(sent), source, sent))
    return max(scored)

# For each eval row, keep the best corpus sentence (the span candidate) and its file.
matches = []
for row in eval_rows:
    score, overlap, _, source, sent = best_span(row["response"])
    matches.append({"overlap": overlap, "source": source, "span": sent,
                    "query": row["instruction"], "expected_answer": row["response"]})

# Normalise text for comparison: lowercase, with single spaces.
def norm(t): return " ".join(t.lower().split())
CORPUS_NORM = norm(" ".join(f.read_text(encoding="utf-8") for f in CORPUS_DIR.glob("*.txt")))
# A span that appears more than once in the corpus cannot say which chunk is the hit, so only spans found once are kept.
# Strongest matches come first.
unique = [m for m in sorted(matches, key=lambda m: -m["overlap"]) if CORPUS_NORM.count(norm(m["span"])) == 1]
# Keep one match per file: the first (strongest) one for each file.
best_per_file = {}
for m in unique:
    best_per_file.setdefault(m["source"], m)
QUERIES = []
# Take the 10 strongest matches. Number them Q1 to Q10 and store the query, the expected answer and the span.
for qid, m in enumerate(sorted(best_per_file.values(), key=lambda m: -m["overlap"])[:10], start=1):
    QUERIES.append({"qid": f"Q{qid}", "source_file": m["source"], "query": m["query"],
                    "expected_answer": m["expected_answer"], "span": m["span"]})

# Check: exactly 10 queries, each from a different file.
assert len(QUERIES) == 10 and len({q["source_file"] for q in QUERIES}) == 10
print(f"eval rows: {len(eval_rows)}, candidate sentences: {len(SENTENCES):,}, queries: {len(QUERIES)}")


**Human check (our choice, not in the brief).** Word overlap can pick a sentence that shares words without answering the question. Each span below was read against its query. Q9 and Q10 matched the wrong file: their answers are lines of the "Lifestyle Goals in Diabetes" list in the ICMR guideline, so those lines are their spans. Read the 10 spans again before the final run, and change `HUMAN_CHECK` if you disagree.

**PDF step (Before you start, p1):** "mark the answer span". This is our human check of the automatic matches above, not a step the brief names.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: applies the human check. "ok" keeps a span. Any other text replaces it.
# Why: word overlap can pick a sentence that shares words without answering the question.
# Output: one line naming the spans that were replaced. Cell 13 shows the 10 queries.
# ---------------------------------------------------------------------------

# One entry per query. "ok" keeps the automatic span. Other text is the new span.
HUMAN_CHECK = {
    "Q1": "ok", "Q2": "ok", "Q3": "ok", "Q4": "ok", "Q5": "ok", "Q6": "ok", "Q7": "ok", "Q8": "ok",
    "Q9": "Advise other behavioural changes for: smoking, other tobacco products and alcohol",
    "Q10": "To individualise the diet according to complications and co-morbidities",
}

# Replace each span that is not "ok". The new text must occur exactly once in the corpus.
for qid, value in HUMAN_CHECK.items():
    if value != "ok":
        q = next(x for x in QUERIES if x["qid"] == qid)
        assert CORPUS_NORM.count(norm(value)) == 1, f"{qid}: the replacement must occur exactly once in the corpus"
        q["span"] = value
        # Point the query at the file that holds the new span.
        q["source_file"] = next(f.name for f in CORPUS_DIR.glob("*.txt") if norm(value) in norm(f.read_text(encoding="utf-8")))

# Every query must have a human-check entry.
assert set(HUMAN_CHECK) == {q["qid"] for q in QUERIES}, "every query needs a human check"
replaced = [qid for qid, value in HUMAN_CHECK.items() if value != "ok"]
print(f"human check: spans replaced for {', '.join(replaced) or 'none'}; the other spans are kept")


**PDF step (Before you start, p1):** "A retrieved chunk is a hit if it contains that span." This cell saves the spans so Parts A, B and C score against the same ones.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: saves the 10 queries to a file and shows them in a table.
# Why: data/answer_spans.json is what the hit rates in Parts A, B and C use.
# Output: data/answer_spans.json and a table of the 10 queries.
# ---------------------------------------------------------------------------

# A chunk is a hit if the span text appears in it. Case and extra spaces are ignored (norm).
def is_hit(chunk_text, span):
    """True if the chunk contains the answer span."""
    return norm(span) in norm(chunk_text)

# Save the queries, so Parts A to D all score against the same spans.
(DATA_DIR / "answer_spans.json").write_text(json.dumps(QUERIES, indent=1, ensure_ascii=False), encoding="utf-8")
show_md_table(["qid", "source file", "query", "expected answer", "answer span"],
              [(q["qid"], q["source_file"], q["query"], q["expected_answer"], q["span"]) for q in QUERIES],
              title="The 10 queries, their expected answers and answer spans", save_as="query_table")


## Part A — Dense search and chunk size (5 marks)

**Goal:** build dense search with FAISS and choose the chunk size for Parts B to D.

**Embedding model.** `all-MiniLM-L6-v2` cuts input at 256 tokens, so 400-token chunks would be truncated. PDF A2 allows this: "either switch to a model with a longer limit or replace that size". We chose a model with a longer limit, as the rule allows: `BAAI/bge-base-en-v1.5` (512-token limit), so both sizes fit. Tokens are counted with this model's tokenizer.

**What we do:**
1. A1.1 — one reusable dense search function (FAISS `IndexFlatIP`).
2. A2.1 — chunk the corpus at 100 and 400 tokens, run the 10 queries, fill the table.
3. A2.2 — choose the chunk size.
4. A1.2 — report the setup for the final chunk size.

*Note on the brief:* the second row of the A2 table says 500, but the A2 text and the submission checklist say 400, so we run 100 and 400.


**PDF step (Before you start, p1):** "Also note your embedding model's maximum input length, for example 256 tokens for all-MiniLM-L6-v2, and count tokens with that model's tokenizer."

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: loads the embedding model and prints its maximum input length and vector size.
# Why: the brief asks us to note the model's maximum input length and to count tokens with its tokenizer.
# Output: the model name, max_seq_length and the vector size.
# ---------------------------------------------------------------------------

from sentence_transformers import SentenceTransformer
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()    # hides harmless warnings, such as the long-input note when A2 tokenizes whole files
hf_logging.disable_progress_bar()   # no "Loading checkpoint shards" bars in the saved notebook

# Our choice. PDF A0 gives all-MiniLM-L6-v2 (256 tokens) only as an example.
# PDF A2 (p1): "either switch to a model with a longer limit or replace that size".
# We chose a model with a longer limit, as the rule allows: bge (512 tokens).
EMBED_MODEL_ID = "BAAI/bge-base-en-v1.5"
EMBED_BATCH = 64   # texts encoded per batch

with working("Loading the embedding model"):
    embed_model = SentenceTransformer(EMBED_MODEL_ID, device=L.DEVICE)
# The model's own tokenizer. A2 uses it to count the tokens in each chunk.
embed_tok = embed_model.tokenizer
# The length of one embedding vector. The brief asks us to report it.
EMBED_DIM = embed_model.get_sentence_embedding_dimension()
print(f"model              : {EMBED_MODEL_ID}")
print(f"max_seq_length     : {embed_model.max_seq_length} tokens")
print(f"vector size        : {EMBED_DIM}")


### A1.1 Dense search function (1 mark)

One function embeds the chunks, builds a FAISS `IndexFlatIP` index and returns the top k chunks for each query. Embeddings are L2-normalised, so the inner product equals cosine similarity. The index is cached per chunk set, so Parts B and C reuse it.

**PDF step A1.1 (p1):** "Write one reusable function that embeds a set of chunks, builds a FAISS IndexFlatIP index (or another vector index) and returns the top k chunks for a query."

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines dense_search(), the one reusable dense search function.
# Why: Parts A, B and C all call it, so every method is searched the same way.
# Output: nothing (definition only). The index build time is printed in A1.2.
# ---------------------------------------------------------------------------

import time

import faiss

# Saved indexes, so the same chunk set is embedded only once.
# Key: (id of the chunk list, its length). Value: (the chunk list, the FAISS index, build seconds).
_INDEX_CACHE = {}


def dense_search(chunks, queries, k):
    """Embed the chunks (once per chunk set), build a FAISS IndexFlatIP, and return the top k chunks for each query.

    Returns one list per query. Each item is a dict with rank (from 1), chunk_id, score and text.
    """
    # The 100-token and 400-token chunk lists are different objects, so each one gets its own index.
    key = (id(chunks), len(chunks))
    if key not in _INDEX_CACHE:
        t0 = time.perf_counter()
        # Embed every chunk. normalize_embeddings makes each vector length 1, so the inner product is the cosine.
        vectors = embed_model.encode([c["text"] for c in chunks], batch_size=EMBED_BATCH,
                                     normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        # IndexFlatIP compares the query with every chunk, so the search is exact (no approximation).
        index = faiss.IndexFlatIP(vectors.shape[1])
        index.add(vectors)
        _INDEX_CACHE[key] = (chunks, index, time.perf_counter() - t0)
    _, index, _ = _INDEX_CACHE[key]

    # Queries get the same normalisation, so their inner product is also the cosine.
    query_vectors = embed_model.encode(queries, batch_size=EMBED_BATCH,
                                       normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    # index.search returns the score and the position of the top k chunks for each query.
    scores, ids = index.search(query_vectors, k)
    results = []
    for qi in range(len(queries)):
        # Ranks start at 1. FAISS writes -1 for an empty slot (when there are fewer than k chunks), so those are skipped.
        results.append([{"rank": r + 1, "chunk_id": chunks[j]["chunk_id"], "score": float(s), "text": chunks[j]["text"]}
                        for r, (s, j) in enumerate(zip(scores[qi], ids[qi])) if j >= 0])
    return results


def index_build_seconds(chunks):
    """Seconds spent embedding the chunks and adding them to the index, for this chunk set."""
    # Build the index first if this chunk set has not been searched yet.
    if (id(chunks), len(chunks)) not in _INDEX_CACHE:
        dense_search(chunks, [QUERIES[0]["query"]], 1)
    return _INDEX_CACHE[(id(chunks), len(chunks))][2]
print("dense_search() defined: embeds the chunks, builds a FAISS IndexFlatIP index, returns the top k.")


### A2.1 Chunk size test (2 marks)

Fixed-size chunks with no overlap, per document, counted with the embedding tokenizer. The text of each chunk is sliced from the original document with the tokenizer's offsets, so the generator sees the original wording.

| Column | How |
| --- | --- |
| Total chunks | number of chunks over the 14 documents |
| Hit rate@3 (%) | share of the 10 queries where one of the top 3 chunks contains the span |
| Tokens sent per query (top 3) | combined `n_tokens` of the top 3, averaged over the 10 queries |

**PDF step A2.1 (p1):** "Chunk your corpus with fixed-size chunking at 100 and 400 tokens, with no overlap."

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines chunk_corpus(), fixed-size token chunking with no overlap.
# Why: the brief asks for 100-token and 400-token chunks, counted with the embedding model's tokenizer.
# Output: nothing (definition only).
# ---------------------------------------------------------------------------

def chunk_corpus(size):
    """Split each .txt file into chunks of `size` tokens, with no overlap.

    Tokens are counted with the embedding tokenizer. Each chunk's text is cut from the original
    file, using the token offsets, so the wording and capitals are kept.
    """
    chunks = []
    for f in sorted(CORPUS_DIR.glob("*.txt")):
        text = f.read_text(encoding="utf-8")
        # offset_mapping gives the start and end character position of each token in the file.
        offsets = embed_tok(text, add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"]
        # Step through the tokens in blocks of `size`. The last block can be shorter.
        for start in range(0, len(offsets), size):
            part = offsets[start:start + size]
            # Cut from the start of the first token to the end of the last token.
            chunks.append({"chunk_id": f"{f.stem}#{start // size}", "doc": f.name,
                           "text": text[part[0][0]:part[-1][1]], "n_tokens": len(part)})
    return chunks
print("chunk_corpus() defined: fixed-size token chunks, no overlap.")


**PDF step A2.1 (p1–2):** "Chunk your corpus with fixed-size chunking at 100 and 400 tokens, with no overlap. Run your dense search from A1 on the 10 queries for each size and fill this table."

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: runs dense search on the 10 queries for each chunk size, and fills the A2 table.
# Why: these numbers are the evidence for choosing the chunk size in A2.2.
# Output: the A2 table (total chunks, hit rate@3, tokens sent per query).
# ---------------------------------------------------------------------------

A2_SIZES = [100, 400]        # the two sizes the A2 text asks for
A2_HIT = {}                  # chunk size -> hit rate@3 (%)
A2_TOKENS = {}               # chunk size -> tokens sent per query (top 3)
chunk_sets = {}              # chunk size -> list of chunks
a2_rows = []
for size in A2_SIZES:
    chunks_for_size = chunk_corpus(size)
    # Check: no chunk is longer than the size we asked for.
    assert max(c["n_tokens"] for c in chunks_for_size) <= size
    chunk_sets[size] = chunks_for_size
    # Look up a chunk by its id, so we can read its token count.
    by_id = {c["chunk_id"]: c for c in chunks_for_size}
    # The top 3 chunks for each of the 10 queries, at this chunk size.
    tops = dense_search(chunks_for_size, [q["query"] for q in QUERIES], 3)
    # A query is a hit if any of its top 3 chunks contains the answer span.
    hits = sum(any(is_hit(r["text"], q["span"]) for r in top) for q, top in zip(QUERIES, tops))
    # Tokens sent: add up the token counts of the 3 chunks for each query, then average over the 10 queries.
    tokens = sum(sum(by_id[r["chunk_id"]]["n_tokens"] for r in top) for top in tops) / len(QUERIES)
    # Hit rate@3 as a percentage of the 10 queries.
    A2_HIT[size] = 100 * hits / len(QUERIES)
    A2_TOKENS[size] = tokens
    a2_rows.append((str(size), f"{len(chunks_for_size):,}", f"{A2_HIT[size]:.0f}", f"{tokens:.0f}"))
print_table(["Chunk size (tokens)", "Total chunks", "Hit rate@3 (%)", "Tokens sent per query (top 3)"],
            a2_rows, title="A2 chunk size test (top 3, 10 queries)", save_as="a2_chunk_size")


### A2.2 Chosen chunk size (1 mark)

*Inference cell below: two or three sentences built from the A2 numbers. Weigh hit rate, tokens per query, and how much context the Assignment 1 model can handle: its adapter was trained with `SFT_MAX_LEN = 512` (Assignment 1, cell 49).*

**PDF step A2.2 (p2):** "In two or three sentences, name the chunk size you will use in Parts B to D and why. Consider hit rate, tokens per query, and how much context your Assignment 1 model can handle."

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: picks the chunk size and counts one Part D prompt with the generator's tokenizer.
# Why: the generator counts tokens differently from the embedding model, so the prompt is counted with its tokenizer.
# Output: the chosen size and the prompt length in tokens.
# ---------------------------------------------------------------------------

from transformers import AutoTokenizer

# Our choice: the size with the highest hit rate@3. A tie goes to the smaller size, because it sends fewer tokens.
# min() picks the lowest key: -hit rate is lowest for the highest hit rate, and the size breaks a tie.
CHUNK_SIZE = min(A2_SIZES, key=lambda s: (-A2_HIT[s], s))
# The chunk list for the chosen size is used again in Parts B, C and D.
chunks = chunk_sets[CHUNK_SIZE]
print(f"chosen chunk size: {CHUNK_SIZE} tokens ({len(chunks):,} chunks)")

# Our check, not in the brief. Count one Part D prompt (3 chunks and a question) with the generator's tokenizer.
# The adapter folder holds the Assignment 1 tokenizer. The layout is close to GEN_PROMPT in D1.1.
mistral_tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)
# The top 3 chunks for the first query, at the chosen size.
top3 = dense_search(chunks, [QUERIES[0]["query"]], 3)[0]
# A prompt-shaped text: the 3 chunks with their numbers, then the question.
prompt_text = "\n".join(f"[{r['rank']}] {r['text']}" for r in top3) + f"\n\nQuestion: {QUERIES[0]['query']}"
# Count with the generator's tokenizer, because the 512-token limit is the generator's.
PROMPT_TOKENS = len(mistral_tok(prompt_text).input_ids)
print(f"Mistral tokens for one prompt (3 chunks and a question): {PROMPT_TOKENS}")
print("adapter training length: 512 tokens (SFT_MAX_LEN, Assignment 1)")


**PDF step A2.2 (p2), written up:** the chosen size and why, from the A2 numbers above. The text is built by code from those numbers.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: writes the A2.2 answer from the numbers in A2.1 and cell 21.
# Why: the brief asks for the chosen size and the reason. Building the text from the numbers keeps them in step.
# Output: one short paragraph.
# ---------------------------------------------------------------------------

from IPython.display import Markdown, display

# Does one Part D prompt fit in the 512 tokens the adapter was trained on?
fits = "within" if PROMPT_TOKENS <= 512 else "above"
display(Markdown(
    f"**A2.2 chosen size: {CHUNK_SIZE} tokens, used in Parts B, C and D.** Hit rate@3 was {A2_HIT[100]:.0f}% at "
    f"100 tokens and {A2_HIT[400]:.0f}% at 400, and the top 3 sent {A2_TOKENS[100]:.0f} and {A2_TOKENS[400]:.0f} "
    f"tokens per query. One Part D prompt at {CHUNK_SIZE} tokens is {PROMPT_TOKENS} Mistral tokens, which is {fits} "
    f"the 512 tokens the Assignment 1 adapter was trained on."))


### A1.2 Dense search setup (1 mark)

Reported for the final chunk set chosen in A2.2.

**PDF step A1.2 (p1):** "Report the embedding model name, vector size, similarity measure, batch size, and how long the index takes to build for your final chunk set."

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: prints the dense search setup for the final chunk set.
# Why: the brief asks for these numbers for the final chunk set.
# Output: a table with the model, vector size, similarity measure, batch size, chunk count and build time.
# ---------------------------------------------------------------------------

print_table(["Setting", "Value"], [
    ("Embedding model", EMBED_MODEL_ID),
    ("Vector size", str(EMBED_DIM)),
    ("Similarity measure", "inner product on L2-normalised vectors (cosine)"),
    ("Batch size", str(EMBED_BATCH)),
    ("Number of chunks", f"{len(chunks):,}"),
    # The build time was stored when the index was built in dense_search.
    ("Index build time (s)", f"{index_build_seconds(chunks):.2f}"),
], title=f"A1.2 dense search setup, chunk size {CHUNK_SIZE} tokens", save_as="a1_2_setup")


## Part B — Keyword and hybrid search (4 marks)

**Goal:** compare dense search, BM25 and a hybrid of the two on the same 10 queries, top 5 each.

**What we do:**
1. B1 — dense search from Part A.
2. B2 — BM25 index, top 5.
3. B3 — hybrid with Reciprocal Rank Fusion over the top 20 of each method:
   `RRF(d) = 1 / (60 + rank in dense) + 1 / (60 + rank in BM25)`. Ranks start at 1; a chunk missing from a list gets nothing from it.
4. B4 — which method works best, with one example query.

| Metric | How |
| --- | --- |
| Query latency (ms) | time to return the top 5 for one query, averaged over the 10 queries, after one warm-up run |
| Hit rate@1 (%) | share of queries whose first result contains the span |
| Hit rate@3 (%) | share of queries with the span in the top 3 |

**PDF step B (p2), the table metrics:** "Query latency (ms): time to return the top 5 for one query, averaged over the 10 queries; skip one warmup run first." Hit rate@1 and @3 are defined in the same table.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines evaluate(), the one helper that scores every method in Parts B and C.
# Why: all methods are timed and scored the same way, so the rows of the table are comparable.
# Output: nothing (definition only).
# ---------------------------------------------------------------------------

def evaluate(search_fn, k=5):
    """Run search_fn(query, k) on the 10 queries, after one warm-up run.

    Returns the average latency (ms), hit rate@1 (%), hit rate@3 (%) and the ranked lists for each query.
    """
    # One warm-up call first, so the first timed query does not include one-time setup. It is not timed.
    search_fn(QUERIES[0]["query"], k)
    latencies, ranked = [], {}
    for q in QUERIES:
        # Time only the search call for this query.
        t0 = time.perf_counter()
        ranked[q["qid"]] = search_fn(q["query"], k)
        latencies.append((time.perf_counter() - t0) * 1000)

    # Hit rate@n: a query counts once if any of its top n results contains the answer span.
    def hit_at(n):
        """Share of queries whose top n results contain the answer span, in %."""
        found = sum(any(is_hit(r["text"], q["span"]) for r in ranked[q["qid"]][:n]) for q in QUERIES)
        return 100 * found / len(QUERIES)

    # The average time per query, the two hit rates, and the ranked lists (kept for the rank comparisons).
    return {"latency_ms": sum(latencies) / len(latencies), "hit1": hit_at(1), "hit3": hit_at(3), "ranked": ranked}
print("evaluate() defined: average latency, hit rate@1 and hit rate@3 for one search method.")


### B1 Dense (1 mark)

**PDF step B1 (p2):** "Dense: run your dense search from Part A on the 10 queries and fill the Dense row."

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: runs dense search (top 5) through evaluate().
# Why: fills the Dense row of the B table.
# Output: the latency, hit rate@1 and hit rate@3 for Dense.
# ---------------------------------------------------------------------------

b_results = {}   # method name -> the result from evaluate()
# Wrap dense search so it takes (query, k), the same as the BM25 and hybrid functions.
dense_fn = lambda q, k: dense_search(chunks, [q], k)[0]
b_results["Dense"] = evaluate(dense_fn)
print(f"Dense: hit rate@1 {b_results['Dense']['hit1']:.0f}%, hit rate@3 {b_results['Dense']['hit3']:.0f}%")


### B2 BM25 (1 mark)

**PDF step B2 (p2):** "BM25: build a BM25 index, retrieve the top 5 for each query, and fill the BM25 row."

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: builds a BM25 index over the chunks and runs it (top 5).
# Why: keyword search matches exact terms, such as drug names and HbA1c, that dense search can miss.
# Output: the latency, hit rate@1 and hit rate@3 for BM25.
# ---------------------------------------------------------------------------

# Our choice, not in the brief: the rank-bm25 library. The brief asks for a BM25 index and names no library.
from rank_bm25 import BM25Okapi


def bm25_tokens(text):
    """Split text into lowercase words and numbers."""
    # Letters and digits stay together, so codes such as hba1c and sglt2 remain one term.
    return re.findall(r"[a-z0-9]+", text.lower())


# Build the BM25 index over the same chunk list. It has one list of tokens per chunk.
bm25 = BM25Okapi([bm25_tokens(c["text"]) for c in chunks])


def bm25_fn(query, k):
    """Top k chunks for one query, in the same format as dense_search."""
    # One BM25 score per chunk for this query.
    scores = bm25.get_scores(bm25_tokens(query))
    # Positions of the highest scores first. The minus sign turns the sort into a descending one.
    top = np.argsort(-scores)[:k]
    return [{"rank": r + 1, "chunk_id": chunks[j]["chunk_id"], "score": float(scores[j]), "text": chunks[j]["text"]}
            for r, j in enumerate(top)]


b_results["BM25"] = evaluate(bm25_fn)
print(f"BM25: hit rate@1 {b_results['BM25']['hit1']:.0f}%, hit rate@3 {b_results['BM25']['hit3']:.0f}%")


### B3 Hybrid with RRF (1 mark)

**PDF step B3 (p2):** "Hybrid: combine the dense and BM25 results with Reciprocal Rank Fusion (RRF), keep the top 5, and fill the Hybrid row." Also: "Fuse the top 20 from each method, not just the top 5."

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines rrf_fuse() and hybrid_fn(), then runs the hybrid search (top 5).
# Why: RRF combines two ranked lists using only the ranks, so the two scores do not need to be on the same scale.
# Output: the Hybrid row.
# ---------------------------------------------------------------------------


def rrf_fuse(dense_ids, bm25_ids, k=60, top=5):
    """Reciprocal rank fusion. Each chunk gets the sum of 1 / (k + rank) over the lists it appears in."""
    score = {}
    for ids in (dense_ids, bm25_ids):
        # Ranks start at 1. A chunk that is missing from a list gets nothing from that list.
        for rank, cid in enumerate(ids, start=1):
            # With k = 60, the gap between ranks matters less, so a chunk that is ranked well by both lists stands out.
            score[cid] = score.get(cid, 0.0) + 1.0 / (k + rank)
    # Highest fused score first, keep the top ones.
    return sorted(score.items(), key=lambda item: -item[1])[:top]


# Look up a chunk by its id.
chunk_by_id = {c["chunk_id"]: c for c in chunks}


def hybrid_fn(query, k):
    """Top 20 from dense and top 20 from BM25, fused with RRF, keep the top k."""
    # Take 20 from each method before fusing, not 5, so a chunk that both methods rank fairly well can still reach the top.
    dense_top = dense_fn(query, 20)
    bm25_top = bm25_fn(query, 20)
    # Fuse the two lists of chunk ids, keeping the top k.
    fused = rrf_fuse([r["chunk_id"] for r in dense_top], [r["chunk_id"] for r in bm25_top], top=k)
    return [{"rank": r + 1, "chunk_id": cid, "score": s, "text": chunk_by_id[cid]["text"]}
            for r, (cid, s) in enumerate(fused)]


b_results["Hybrid (RRF)"] = evaluate(hybrid_fn)
print(f"Hybrid (RRF): hit rate@1 {b_results['Hybrid (RRF)']['hit1']:.0f}%, "
      f"hit rate@3 {b_results['Hybrid (RRF)']['hit3']:.0f}%")


**PDF step B (p2), the table:** "Fill this table for all three methods." The cell below prints that table.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: prints the Part B comparison table, with a row for each method.
# Why: the brief asks for one table with Dense, BM25 and Hybrid filled in.
# Output: the Part B table.
# ---------------------------------------------------------------------------


def rank_of(method, qid, span):
    """Rank of the first top-5 chunk that contains the span. Returns "-" if none does."""
    # Walk down the ranked list, so the first match is the best rank.
    for r in b_results[method]["ranked"][qid]:
        if is_hit(r["text"], span):
            return str(r["rank"])
    return "-"


print_table(["Method", "Average query latency (ms)", "Hit rate@1 (%)", "Hit rate@3 (%)"],
            [(m, f"{v['latency_ms']:.1f}", f"{v['hit1']:.0f}", f"{v['hit3']:.0f}") for m, v in b_results.items()],
            title=f"Part B search comparison (chunk size {CHUNK_SIZE}, top 5)", save_as="b_comparison")


### B4 Best method (1 mark)

**PDF step B4 (p3):** "In two or three sentences, say which method works best for your queries and why, using one query as an example."

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: picks the best method, chooses an example query, and writes the B4 answer.
# Why: the brief asks which method works best and why, with one query as the example.
# Output: one paragraph, built from the Part B numbers.
# ---------------------------------------------------------------------------

from IPython.display import Markdown, display

# Our choice of "best": the highest hit rate@1, then the highest hit rate@3, then the faster method.
# The tuple below is compared in that order. -latency is the largest for the fastest method.
BEST_METHOD = max(b_results, key=lambda m: (b_results[m]["hit1"], b_results[m]["hit3"], -b_results[m]["latency_ms"]))
# One short reason per method, for the text.
REASON = {
    "Dense": "dense search matches meaning, so it finds answers that are worded differently from the question",
    "BM25": "BM25 matches exact words such as drug names and numbers, which dense search can blur",
    "Hybrid (RRF)": "the hybrid keeps a chunk that either list ranks well, so it recovers answers that one method misses",
}


def _rank_or_last(method, q):
    """The rank of the answer chunk, or 99 if the method does not return it in its top 5."""
    r = rank_of(method, q["qid"], q["span"])
    return int(r) if r.isdigit() else 99


# Our choice of example: a query where the best method ranks the answer chunk strictly higher than every other method.
# A missing chunk counts as rank 99, so it is the worst. If no query does this, use the first query where the ranks differ.
ex = next((q for q in QUERIES
           if _rank_or_last(BEST_METHOD, q) < min(_rank_or_last(m, q) for m in b_results if m != BEST_METHOD)), None)
if ex is None:
    ex = next((q for q in QUERIES if len({rank_of(m, q["qid"], q["span"]) for m in b_results}) > 1), QUERIES[0])
# The best method's numbers, used in the text.
best = b_results[BEST_METHOD]
others = " and ".join(f"{m} {b_results[m]['hit1']:.0f}% / {b_results[m]['hit3']:.0f}%" for m in b_results if m != BEST_METHOD)
# Other methods with the same hit rates as the best one. If there are any, the text says it was chosen for speed.
tied = [m for m in b_results if m != BEST_METHOD and (b_results[m]["hit1"], b_results[m]["hit3"]) == (best["hit1"], best["hit3"])]
tie_note = (f" It ties with {' and '.join(tied)} on both hit rates and is picked because it is faster "
            f"({best['latency_ms']:.1f} ms per query)." if tied else "")
display(Markdown(
    f"**B4 best method: {BEST_METHOD}.** Hit rate@1 / hit rate@3: {best['hit1']:.0f}% / {best['hit3']:.0f}%, "
    f"against {others}.{tie_note} Why: {REASON[BEST_METHOD]}. Example query {ex['qid']}: "
    f"\"{ex['query']}\" The answer chunk ranks {rank_of('Dense', ex['qid'], ex['span'])} with Dense, "
    f"{rank_of('BM25', ex['qid'], ex['span'])} with BM25 and {rank_of('Hybrid (RRF)', ex['qid'], ex['span'])} "
    f"with Hybrid (RRF)."))


## Part C — Reranking (4 marks)

**Goal:** re-score the top 10 of the best Part B method with a cross-encoder and keep the top 3.

**What we do:**
1. C1 — retrieve the top 10 with the best method; re-score with `cross-encoder/ms-marco-MiniLM-L-6-v2`; keep the top 3.
2. C2 — show the top 3 before and after reranking for each query, and the rank change rate.
3. C3 — reranking time per query and hit rate@1 before and after.
4. C4 — did reranking help?

**Rank change rate** = share of the 10 queries whose first result changes after reranking.

### C1 Rerank (1 mark)

**PDF step C1 (p3):** "Rerank: take your best method from Part B and retrieve the top 10 for each query. Re-score them with a cross-encoder, for example cross-encoder/ms-marco-MiniLM-L-6-v2, and keep the top 3."

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: loads the cross-encoder, then reranks the top 10 of the best Part B method for each query.
# Why: a cross-encoder reads the query and a chunk together, so it can order them more accurately than the first search.
# Output: the device used, and the top 3 chunks for each query after reranking.
# ---------------------------------------------------------------------------

import torch
from sentence_transformers import CrossEncoder

# PDF C1 says "for example" this model, so any cross-encoder is allowed. We chose this one.
RERANK_MODEL_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"
with working("Loading the cross-encoder"):
    ce = CrossEncoder(RERANK_MODEL_ID, device=L.DEVICE)
print(f"cross-encoder: {RERANK_MODEL_ID} on {L.DEVICE}")

# Pick the search function for the best method from Part B, so reranking starts from that method.
best_fn = {"Dense": dense_fn, "BM25": bm25_fn, "Hybrid (RRF)": hybrid_fn}[BEST_METHOD]


def rerank_top3(query):
    """Rerank the best method's top 10 for one query. Returns (top 3 before, top 3 after, seconds spent)."""
    # The top 10 from the best method. These are the candidates to rerank.
    top10 = best_fn(query, 10)
    # One (query, chunk) pair per candidate. The cross-encoder scores them in one batch.
    pairs = [(query, c["text"]) for c in top10]
    t0 = time.perf_counter()
    scores = ce.predict(pairs)
    if L.IS_GPU:
        # GPU work runs in the background, so wait for it to finish before stopping the timer.
        torch.cuda.synchronize()
    seconds = time.perf_counter() - t0
    # Positions of the 10 candidates, sorted by cross-encoder score (highest first). Keep the top 3.
    order = sorted(range(len(top10)), key=lambda i: -scores[i])[:3]
    # The new top 3, with their new rank and the cross-encoder score.
    after = [dict(top10[i], rank=r + 1, ce_score=float(scores[i])) for r, i in enumerate(order)]
    return top10[:3], after, seconds


# One warm-up call, so the first timed call does not include one-time setup. It is not timed.
ce.predict([(QUERIES[0]["query"], "warm-up")])
# Rerank each query. "before" is the best method's own top 3, kept for the comparison in C2.
final_chunks, before_top3, rerank_ms = {}, {}, []
for q in QUERIES:
    before, after, seconds = rerank_top3(q["query"])
    before_top3[q["qid"]] = before
    final_chunks[q["qid"]] = after
    rerank_ms.append(seconds * 1000)
print(f"reranked {len(QUERIES)} queries; kept the top 3 of 10 for each")


### C2 Compare rankings (1 mark)

**PDF step C2 (p3):** "Compare rankings: for each query, show the top 3 before and after reranking, and report the rank change rate." The rate is defined in the same step: "the share of the 10 queries whose first result changes after reranking."

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: prints the top 3 before and after reranking for each query, and counts the rank changes.
# Why: the brief asks to see the change for every query, and to report the rank change rate.
# Output: the before and after top 3 for each query, and the rank change rate.
# ---------------------------------------------------------------------------

changed = 0
for q in QUERIES:
    qid = q["qid"]
    print(f"\n{qid}  {q['query']}")
    print("  Before reranking (best method's own top 3):")
    for r in before_top3[qid]:
        # HIT marks a chunk that contains the answer span.
        mark = "HIT " if is_hit(r["text"], q["span"]) else "    "
        # Each chunk is printed on one line, with the first 80 characters of its text.
        print(f"    {r['rank']}. {mark}{r['chunk_id']:<30} {' '.join(r['text'].split())[:80]}")
    print("  After reranking (cross-encoder top 3):")
    for r in final_chunks[qid]:
        mark = "HIT " if is_hit(r["text"], q["span"]) else "    "
        print(f"    {r['rank']}. {mark}{r['chunk_id']:<30} {' '.join(r['text'].split())[:80]}")
    # A query counts as changed if its first chunk is a different chunk after reranking.
    if before_top3[qid][0]["chunk_id"] != final_chunks[qid][0]["chunk_id"]:
        changed += 1

RANK_CHANGE = 100 * changed / len(QUERIES)   # % of queries whose first result changed
print(f"\nRank change rate: {RANK_CHANGE:.0f}% of queries (first result changed)")


### C3 Measure (1 mark)

**PDF step C3 (p3):** "Measure: report the reranking time per query and the hit rate@1 after reranking, and fill this table."

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: fills the reranking table.
# Why: the table sets the hit rate@1 and the time before and after reranking side by side, so the cost can be weighed against the gain.
# Output: the reranking time per query, the rank change rate, and hit rate@1 before and after.
# ---------------------------------------------------------------------------

# Hit rate@1 before reranking: the same number as the best method in Part B.
hit1_before = b_results[BEST_METHOD]["hit1"]
# Hit rate@1 after reranking: share of queries whose new first chunk contains the span.
hit1_after = 100 * sum(is_hit(final_chunks[q["qid"]][0]["text"], q["span"]) for q in QUERIES) / len(QUERIES)
# Average reranking time per query, in ms.
RERANK_MS = sum(rerank_ms) / len(rerank_ms)
print_table(["Measure", "Value"], [
    ("Reranking time per query (ms)", f"{RERANK_MS:.1f}"),
    ("Rank change rate (%)", f"{RANK_CHANGE:.0f}"),
    ("Hit rate@1 before reranking (from Part B, same method)", f"{hit1_before:.0f}"),
    ("Hit rate@1 after reranking", f"{hit1_after:.0f}"),
], title="Part C reranking measures", save_as="c_measures")


### C4 Did reranking help? (1 mark)

**PDF step C4 (p3):** "In two or three sentences, say whether reranking helped in your domain, using the hit rates before and after and the time it added."

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: writes the C4 answer from the hit rates and the time, using the C3 numbers.
# Why: the brief asks whether reranking helped, using the hit rates before and after and the time it added.
# Output: one short paragraph.
# ---------------------------------------------------------------------------

from IPython.display import Markdown, display

# Compare hit rate@1 before and after reranking, to pick the verdict wording.
if hit1_after > hit1_before:
    verdict = "reranking helped"
elif hit1_after == hit1_before:
    verdict = "reranking made no difference"
else:
    verdict = "reranking lowered hit rate@1"
# If reranking did not help, the text adds a possible reason. The cross-encoder was trained on web search, not clinical text.
note = ("" if hit1_after > hit1_before else
        " The cross-encoder was trained on web search (MS MARCO), not clinical text, which may explain this.")
display(Markdown(
    f"**C4.** Hit rate@1 went from {hit1_before:.0f}% to {hit1_after:.0f}% for +{RERANK_MS:.0f} ms per query, "
    f"so {verdict}.{note} "
    f"Weighing accuracy against time: the hit rate@1 gain of {hit1_after - hit1_before:.0f} points costs {RERANK_MS:.0f} ms per query."))


## Part D — Generate and check answers (7 marks)

### D1 Generate grounded answers (3 marks)

**Goal:** answer the 10 queries with the Assignment 1 model, using only the 3 reranked chunks, with citations and a "Not found" reply.

**What we do:**
The brief has three D1 steps, one mark each:
1. D1.1 — the prompt (1 mark): answer only from the chunks, cite them, and say "Not found in the documents".
2. D1.2 — all 10 answers (1 mark), each from the 3 reranked chunks.
3. D1.3 — one query whose answer is not in the corpus (1 mark).

**Generator:** the Assignment 1 model only (Mistral-7B-v0.1 + CPT weights + QLoRA adapter). It answers the 10 queries and the out-of-corpus query.


**PDF step (Overview, p1):** "your fine-tuned model as the answer generator". This cell loads that model: the Assignment 1 CPT model with the QLoRA adapter.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: loads the Assignment 1 generator, which is the CPT model in 4-bit with the QLoRA adapter on top.
# Why: the brief asks us to reuse the Assignment 1 model as the answer generator.
# The loading settings are copied from Assignment 1 cells 50 and 51, so the model is loaded the way it was trained.
# Output: the generator name.
# ---------------------------------------------------------------------------

import torch
import transformers
from packaging.version import Version
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# The base model that Assignment 1 started from.
GEN_BASE_ID = "mistralai/Mistral-7B-v0.1"   # Assignment 1 MODEL_ID (cell 18). The brief calls it "your fine-tuned model".
GENERATOR_NAME = f"{GEN_BASE_ID} + CPT + QLoRA adapter (Assignment 1)"
# Our choice, not in the brief. Copied from Assignment 1 cell 51 (SFT_MAX_NEW_TOKENS). Answers are 1 to 3 sentences.
GEN_MAX_NEW_TOKENS = 120
# Our choice, not in the brief. The Assignment 1 answer ends where the next "Instruction:" starts (cell 51).
GEN_STOPS = ["Instruction:"]

# The name of the dtype argument changed in transformers 4.56 (as in Assignment 1 cell 18).
DTYPE_KW = "dtype" if Version(transformers.__version__) >= Version("4.56") else "torch_dtype"
# 4-bit loading needs a CUDA GPU (Assignment 1 cell 50 does the same check).
if not L.IS_GPU:
    raise RuntimeError("4-bit loading requires a CUDA GPU. Run this notebook on the lab GPU machine.")
# The compute dtype is the setup cell's dtype, which is bf16 on the A100 (Assignment 1 cell 50).
COMPUTE_DTYPE = getattr(torch, L.DTYPE) if L.DTYPE in ("bfloat16", "float16") else torch.float16
# The 4-bit settings for QLoRA, copied from Assignment 1 cell 50. The adapter was trained on this 4-bit base.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                       # store the weights in 4 bits
    bnb_4bit_quant_type="nf4",               # the NF4 data type
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,    # do the math in bf16
    bnb_4bit_use_double_quant=True,          # quantise the scaling factors too, to save memory
)

with working("Loading the generator (CPT model in 4-bit, then the adapter)", folder=CPT_DIR):
    # Load the CPT checkpoint in 4-bit, onto GPU 0 (Assignment 1 cell 51).
    gen_base = AutoModelForCausalLM.from_pretrained(CPT_DIR, quantization_config=bnb_config,
                                                    **{DTYPE_KW: COMPUTE_DTYPE}, device_map={"": 0})
    # Add the QLoRA adapter on top of the CPT model. eval() turns off dropout, because we only answer here.
    gen_model = PeftModel.from_pretrained(gen_base, ADAPTER_DIR).eval()
# The tokenizer saved with the adapter. It has the Instruction and Response chat template from Assignment 1 cell 49.
gen_tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)
print(f"generator: {GENERATOR_NAME}")


#### D1.1 Generation prompt (1 mark)

**PDF step D1.1 (p3):** "Write a prompt that tells the model to answer only from the chunks, cite the chunk numbers it used, for example [2], and reply "Not found in the documents" when the chunks do not contain the answer. Show the prompt."

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines the generation prompt and prints it, so the prompt can be read.
# Why: the brief asks to show the prompt. The three rules are: answer only from the chunks, cite them, and say "Not found".
# Output: the prompt template.
# ---------------------------------------------------------------------------

# The prompt template. generate_answer() fills in the three chunks and the question.
GEN_PROMPT = """Answer the question using ONLY the numbered chunks below.
Cite the chunk number you used after each fact, for example [2]. Use only the numbers 1, 2 and 3.
Stop after your answer. Do not write a new question or a new instruction.
If the chunks do not contain the answer, reply exactly: Not found in the documents

Chunks:
[1] {chunk_1}
[2] {chunk_2}
[3] {chunk_3}

Question: {question}"""

print(GEN_PROMPT)


**PDF step D1.2 (p3):** "For each of the 10 queries, give your Assignment 1 model the top 3 chunks after reranking, each labelled with a number, and show all 10 answers." This cell defines the helpers that the 10 answer cells use.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines the answer helpers, and the checks for citations and the Not found reply.
# Why: the same code answers the 10 queries and the out-of-corpus query, so every answer goes through the same steps.
# Output: nothing (definitions only).
# ---------------------------------------------------------------------------

NOT_FOUND = "Not found in the documents"
# A citation is a number in square brackets, such as [2].
CITE = re.compile(r"\[(\d+)\]")


def clean_answer(raw):
    """Cut at the next "Instruction:", then keep the text after "Response:", as Assignment 1 cell 51 does."""
    # Cut the raw text where the next "Instruction:" starts, so a made-up next question is dropped.
    text = raw
    for stop in GEN_STOPS:
        text = text.split(stop)[0]
    # If the model repeats "Response:", keep only the text after it.
    if "Response:" in text:
        text = text.split("Response:")[1]
    return text.strip()


def generate_answer(question, chunk_texts):
    """Answer one question from its 3 chunks, in the Assignment 1 format."""
    # Put the question and the 3 chunks into the prompt template, in the order given.
    content = GEN_PROMPT.format(chunk_1=chunk_texts[0], chunk_2=chunk_texts[1], chunk_3=chunk_texts[2], question=question)
    # Wrap the text in the model's chat format, so it is read as a user turn.
    prompt = gen_tok.apply_chat_template([{"role": "user", "content": content}],
                                         tokenize=False, add_generation_prompt=True).rstrip()
    # Turn the prompt into token ids and move them to the GPU.
    ids = gen_tok(prompt, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        # Greedy decoding (do_sample=False): the same answer on every run.
        gen = gen_model.generate(**ids, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=False,
                                 pad_token_id=gen_tok.eos_token_id)
    # Decode only the new tokens, not the prompt.
    raw = gen_tok.decode(gen[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    return clean_answer(raw)


def is_not_found(answer):
    """True if the answer is the Not found sentence. A final full stop is allowed."""
    # Remove the spaces and one final full stop, then compare with the exact sentence.
    return answer.strip().rstrip(".").strip() == NOT_FOUND


def has_valid_citation(answer):
    """True if the answer cites at least one chunk, and every number it cites is 1, 2 or 3."""
    # All the numbers the answer cites. Each one must be 1, 2 or 3.
    nums = [int(n) for n in CITE.findall(answer)]
    return bool(nums) and all(1 <= n <= 3 for n in nums)
print("generate_answer() defined: greedy answer from the 3 numbered chunks, with the citation and Not found checks.")


#### D1.2 All 10 answers (1 mark)

**PDF step D1.2 (p3):** "For each of the 10 queries, give your Assignment 1 model the top 3 chunks after reranking, each labelled with a number, and show all 10 answers." This cell answers Q1.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers query Q1 from its 3 reranked chunks with the Assignment 1 model, then shows the answer.
# Why: one cell per query, so each answer and the chunks it cites can be read on its own.
# Output: the query, the answer and the chunk numbers the answer cites.
# ---------------------------------------------------------------------------

answers = {}   # qid -> answer. The 10 query cells fill it, and the judge in D2 reads it.

QID = "Q1"
# Find this query in QUERIES by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# Answer from this query's 3 reranked chunks. Decoding is greedy, so the answer is the same on every run.
answers[QID] = generate_answer(q["query"], [c["text"] for c in final_chunks[QID]])
print(f"{QID}  {q['query']}")
print(f"  Answer: {answers[QID]}")
# The chunk numbers the answer cites, sorted, with repeats removed. '-' means none.
print(f"  Chunks cited: {','.join(sorted(set(CITE.findall(answers[QID])))) or '-'}")


**PDF step D1.2 (p3):** "For each of the 10 queries, give your Assignment 1 model the top 3 chunks after reranking, each labelled with a number, and show all 10 answers." This cell answers Q2.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers query Q2 from its 3 reranked chunks with the Assignment 1 model, then shows the answer.
# Why: one cell per query, so each answer and the chunks it cites can be read on its own.
# Output: the query, the answer and the chunk numbers the answer cites.
# ---------------------------------------------------------------------------

QID = "Q2"
# Find this query in QUERIES by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# Answer from this query's 3 reranked chunks. Decoding is greedy, so the answer is the same on every run.
answers[QID] = generate_answer(q["query"], [c["text"] for c in final_chunks[QID]])
print(f"{QID}  {q['query']}")
print(f"  Answer: {answers[QID]}")
# The chunk numbers the answer cites, sorted, with repeats removed. '-' means none.
print(f"  Chunks cited: {','.join(sorted(set(CITE.findall(answers[QID])))) or '-'}")


**PDF step D1.2 (p3):** "For each of the 10 queries, give your Assignment 1 model the top 3 chunks after reranking, each labelled with a number, and show all 10 answers." This cell answers Q3.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers query Q3 from its 3 reranked chunks with the Assignment 1 model, then shows the answer.
# Why: one cell per query, so each answer and the chunks it cites can be read on its own.
# Output: the query, the answer and the chunk numbers the answer cites.
# ---------------------------------------------------------------------------

QID = "Q3"
# Find this query in QUERIES by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# Answer from this query's 3 reranked chunks. Decoding is greedy, so the answer is the same on every run.
answers[QID] = generate_answer(q["query"], [c["text"] for c in final_chunks[QID]])
print(f"{QID}  {q['query']}")
print(f"  Answer: {answers[QID]}")
# The chunk numbers the answer cites, sorted, with repeats removed. '-' means none.
print(f"  Chunks cited: {','.join(sorted(set(CITE.findall(answers[QID])))) or '-'}")


**PDF step D1.2 (p3):** "For each of the 10 queries, give your Assignment 1 model the top 3 chunks after reranking, each labelled with a number, and show all 10 answers." This cell answers Q4.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers query Q4 from its 3 reranked chunks with the Assignment 1 model, then shows the answer.
# Why: one cell per query, so each answer and the chunks it cites can be read on its own.
# Output: the query, the answer and the chunk numbers the answer cites.
# ---------------------------------------------------------------------------

QID = "Q4"
# Find this query in QUERIES by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# Answer from this query's 3 reranked chunks. Decoding is greedy, so the answer is the same on every run.
answers[QID] = generate_answer(q["query"], [c["text"] for c in final_chunks[QID]])
print(f"{QID}  {q['query']}")
print(f"  Answer: {answers[QID]}")
# The chunk numbers the answer cites, sorted, with repeats removed. '-' means none.
print(f"  Chunks cited: {','.join(sorted(set(CITE.findall(answers[QID])))) or '-'}")


**PDF step D1.2 (p3):** "For each of the 10 queries, give your Assignment 1 model the top 3 chunks after reranking, each labelled with a number, and show all 10 answers." This cell answers Q5.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers query Q5 from its 3 reranked chunks with the Assignment 1 model, then shows the answer.
# Why: one cell per query, so each answer and the chunks it cites can be read on its own.
# Output: the query, the answer and the chunk numbers the answer cites.
# ---------------------------------------------------------------------------

QID = "Q5"
# Find this query in QUERIES by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# Answer from this query's 3 reranked chunks. Decoding is greedy, so the answer is the same on every run.
answers[QID] = generate_answer(q["query"], [c["text"] for c in final_chunks[QID]])
print(f"{QID}  {q['query']}")
print(f"  Answer: {answers[QID]}")
# The chunk numbers the answer cites, sorted, with repeats removed. '-' means none.
print(f"  Chunks cited: {','.join(sorted(set(CITE.findall(answers[QID])))) or '-'}")


**PDF step D1.2 (p3):** "For each of the 10 queries, give your Assignment 1 model the top 3 chunks after reranking, each labelled with a number, and show all 10 answers." This cell answers Q6.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers query Q6 from its 3 reranked chunks with the Assignment 1 model, then shows the answer.
# Why: one cell per query, so each answer and the chunks it cites can be read on its own.
# Output: the query, the answer and the chunk numbers the answer cites.
# ---------------------------------------------------------------------------

QID = "Q6"
# Find this query in QUERIES by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# Answer from this query's 3 reranked chunks. Decoding is greedy, so the answer is the same on every run.
answers[QID] = generate_answer(q["query"], [c["text"] for c in final_chunks[QID]])
print(f"{QID}  {q['query']}")
print(f"  Answer: {answers[QID]}")
# The chunk numbers the answer cites, sorted, with repeats removed. '-' means none.
print(f"  Chunks cited: {','.join(sorted(set(CITE.findall(answers[QID])))) or '-'}")


**PDF step D1.2 (p3):** "For each of the 10 queries, give your Assignment 1 model the top 3 chunks after reranking, each labelled with a number, and show all 10 answers." This cell answers Q7.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers query Q7 from its 3 reranked chunks with the Assignment 1 model, then shows the answer.
# Why: one cell per query, so each answer and the chunks it cites can be read on its own.
# Output: the query, the answer and the chunk numbers the answer cites.
# ---------------------------------------------------------------------------

QID = "Q7"
# Find this query in QUERIES by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# Answer from this query's 3 reranked chunks. Decoding is greedy, so the answer is the same on every run.
answers[QID] = generate_answer(q["query"], [c["text"] for c in final_chunks[QID]])
print(f"{QID}  {q['query']}")
print(f"  Answer: {answers[QID]}")
# The chunk numbers the answer cites, sorted, with repeats removed. '-' means none.
print(f"  Chunks cited: {','.join(sorted(set(CITE.findall(answers[QID])))) or '-'}")


**PDF step D1.2 (p3):** "For each of the 10 queries, give your Assignment 1 model the top 3 chunks after reranking, each labelled with a number, and show all 10 answers." This cell answers Q8.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers query Q8 from its 3 reranked chunks with the Assignment 1 model, then shows the answer.
# Why: one cell per query, so each answer and the chunks it cites can be read on its own.
# Output: the query, the answer and the chunk numbers the answer cites.
# ---------------------------------------------------------------------------

QID = "Q8"
# Find this query in QUERIES by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# Answer from this query's 3 reranked chunks. Decoding is greedy, so the answer is the same on every run.
answers[QID] = generate_answer(q["query"], [c["text"] for c in final_chunks[QID]])
print(f"{QID}  {q['query']}")
print(f"  Answer: {answers[QID]}")
# The chunk numbers the answer cites, sorted, with repeats removed. '-' means none.
print(f"  Chunks cited: {','.join(sorted(set(CITE.findall(answers[QID])))) or '-'}")


**PDF step D1.2 (p3):** "For each of the 10 queries, give your Assignment 1 model the top 3 chunks after reranking, each labelled with a number, and show all 10 answers." This cell answers Q9.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers query Q9 from its 3 reranked chunks with the Assignment 1 model, then shows the answer.
# Why: one cell per query, so each answer and the chunks it cites can be read on its own.
# Output: the query, the answer and the chunk numbers the answer cites.
# ---------------------------------------------------------------------------

QID = "Q9"
# Find this query in QUERIES by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# Answer from this query's 3 reranked chunks. Decoding is greedy, so the answer is the same on every run.
answers[QID] = generate_answer(q["query"], [c["text"] for c in final_chunks[QID]])
print(f"{QID}  {q['query']}")
print(f"  Answer: {answers[QID]}")
# The chunk numbers the answer cites, sorted, with repeats removed. '-' means none.
print(f"  Chunks cited: {','.join(sorted(set(CITE.findall(answers[QID])))) or '-'}")


**PDF step D1.2 (p3):** "For each of the 10 queries, give your Assignment 1 model the top 3 chunks after reranking, each labelled with a number, and show all 10 answers." This cell answers Q10.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers query Q10 from its 3 reranked chunks with the Assignment 1 model, then shows the answer.
# Why: one cell per query, so each answer and the chunks it cites can be read on its own.
# Output: the query, the answer and the chunk numbers the answer cites.
# ---------------------------------------------------------------------------

QID = "Q10"
# Find this query in QUERIES by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# Answer from this query's 3 reranked chunks. Decoding is greedy, so the answer is the same on every run.
answers[QID] = generate_answer(q["query"], [c["text"] for c in final_chunks[QID]])
print(f"{QID}  {q['query']}")
print(f"  Answer: {answers[QID]}")
# The chunk numbers the answer cites, sorted, with repeats removed. '-' means none.
print(f"  Chunks cited: {','.join(sorted(set(CITE.findall(answers[QID])))) or '-'}")


#### D1.3 A query whose answer is not in the corpus (1 mark)

**PDF step D1.3 (p3):** "Add one query whose answer is not in your corpus, and show that the model replies that it cannot find it."

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: runs one question that the corpus cannot answer, through the same retrieval, reranking and prompt.
# Why: shows whether the generator replies "Not found in the documents" instead of guessing.
# Output: the question, its 3 chunks, the reply, and whether the reply is the Not found sentence.
# ---------------------------------------------------------------------------

# Our choice: a question that the 14 diabetes documents cannot answer.
OOC_QUERY = "Who won the 2018 FIFA World Cup?"
# The same retrieval and reranking as Part C. The first value (the candidate list) is not needed here, so it is named _.
_, ooc_top3, _ = rerank_top3(OOC_QUERY)
# Print the 3 chunks the generator will see, with their numbers.
for i, c in enumerate(ooc_top3, 1):
    print(f"[{i}] {c['chunk_id']}: {c['text'][:120]}")
# The same generator and prompt as for the 10 queries, so the Not found check is fair.
ooc_answer = generate_answer(OOC_QUERY, [c["text"] for c in ooc_top3])
print(f"\nQuestion: {OOC_QUERY}")
print(f"Reply: {ooc_answer}")
# Check the reply with the same exact-match rule as the 10 answers.
print(f"Replied Not found: {'yes' if is_not_found(ooc_answer) else 'no'}")


#### D1 switch to an instruct model (not used in this notebook)

PDF D1 (p4): "If your Assignment 1 model ignores the chunks or never says 'Not found', you may switch to an instruct model such as Llama 3.1 8B Instruct or Qwen2.5 7B Instruct. Name the model you used and show one example of the problem you saw."

This notebook answers all 10 queries and the out-of-corpus query with the Assignment 1 model only, so no switch is made.


### D2 Check the answers with an LLM judge (4 marks)

**Goal:** grade each answer for correctness against the expected answer and for support by its 3 chunks.

**Judge model:** different from the generator. The judge is `Qwen/Qwen2.5-7B-Instruct`, which differs from the generator (the Assignment 1 Mistral model). Greedy decoding (temperature 0).

**What we do:**
1. D2.1 — run the judge on the 10 answers; per-query table and summary counts.
2. D2.2 — spot-check 3 verdicts by hand.
3. D2.3 — find the cause of one wrong or unsupported answer.
4. D2.4 — suggest one fix.


**PDF step D2 (p4):** "Use a judge model different from your generator, for example Llama 3.1 8B Instruct or Qwen2.5 7B Instruct, and name it." This cell loads the judge.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: frees the generator from GPU memory, then loads the judge model.
# Why: the judge runs after the generator is finished, so the GPU memory is free for it.
# Output: the judge name.
# ---------------------------------------------------------------------------

import gc

# Delete the generator, run the garbage collector, and empty the GPU cache. Then the judge can load.
del gen_model, gen_base
gc.collect()
torch.cuda.empty_cache()

# Our choice: Qwen2.5 7B Instruct. The brief lists it as an example. It is a different model from the generator.
JUDGE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
# Stop early if the judge and the generator were set to the same model.
assert JUDGE_MODEL_ID not in GENERATOR_NAME, "the judge must be a different model from the generator"
with working("Loading the judge"):
    judge_tok = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
    judge_model = AutoModelForCausalLM.from_pretrained(JUDGE_MODEL_ID, device_map={"": 0}).eval()
# Qwen's own generation settings include sampling values. Greedy decoding does not use them, so they are cleared to avoid warnings.
judge_model.generation_config.temperature = None
judge_model.generation_config.top_p = None
judge_model.generation_config.top_k = None


def judge_generate(prompt_text):
    """Greedy reply from the judge (temperature 0), using the judge's own chat template."""
    # Wrap the prompt in the judge's chat format, so it is read as one user turn.
    text = judge_tok.apply_chat_template([{"role": "user", "content": prompt_text}],
                                         tokenize=False, add_generation_prompt=True)
    ids = judge_tok(text, return_tensors="pt").to(judge_model.device)
    with torch.no_grad():
        # Greedy decoding. The judge's reply is at most 120 new tokens.
        out = judge_model.generate(**ids, max_new_tokens=120, do_sample=False, pad_token_id=judge_tok.eos_token_id)
    # Decode only the new tokens, not the prompt.
    return judge_tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)


print(f"judge: {JUDGE_MODEL_ID}")


#### D2.1 Judge prompt and results (1 mark)


**PDF step D2.1 (p4):** "Write and run the judge: write one judge prompt that checks two things for each answer: Correct ... Supported ... Show your prompt." The hints below it are also in the prompt.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines the judge prompt and prints it, so it can be read.
# Why: the brief asks us to show the prompt. It has two checks (CORRECT and SUPPORTED), and each label is defined.
# Output: the judge prompt.
# ---------------------------------------------------------------------------

# The judge prompt. The placeholders are filled in for each query in the cells below.
JUDGE_PROMPT = """You are grading one answer about Type 2 Diabetes.

Question: {question}
Expected answer (reference): {expected}
Retrieved chunks:
[1] {chunk_1}
[2] {chunk_2}
[3] {chunk_3}
Model answer: {answer}

Check 1, CORRECT: compare the model answer with the expected answer.
  CORRECT = it states the same key fact as the expected answer.
  PARTLY CORRECT = it states only part of the key fact, or the right fact with a detail missing or wrong.
  WRONG = it states a different fact, contradicts the expected answer, or says "Not found in the documents".
Check 2, SUPPORTED: can every statement in the model answer be found in the 3 chunks above?
  YES = every statement is in the chunks. NO = at least one statement is not.
  A "Not found in the documents" reply makes no claims, so it is SUPPORTED: YES.
  A reply of exactly "Not found in the documents" always gets CORRECT: WRONG and SUPPORTED: YES.
REASON: one sentence. If SUPPORTED is NO, the reason must quote the unsupported statement in double quotes.

Reply in exactly this format, with nothing else:
CORRECT: <CORRECT | PARTLY CORRECT | WRONG>
SUPPORTED: <YES | NO>
REASON: <one sentence>"""
print(JUDGE_PROMPT)


**PDF step D2.1 (p4):** "Show your prompt, run it on all 10 answers, show the results in one table with a row per query (query, correct, supported, reason)." This cell reads the judge's reply into those fields.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines parse_verdict(), which reads the judge's three labelled lines.
# Why: the judge replies in the fixed format from D2.1, so the code can count the verdicts.
# Output: nothing (definitions only).
# ---------------------------------------------------------------------------


def parse_verdict(text):
    """Read the CORRECT, SUPPORTED and REASON labels. Remove the < > and * marks the judge may copy from the template."""
    # Remove the < >, and * marks, so that "<CORRECT>" reads as CORRECT.
    flat = re.sub(r"[<>*]", "", text)

    # Read the text after "label:", up to the next label or the end of the line.
    def pick(label):
        m = re.search(rf"{label}\s*:\s*(.*?)(?=\s+(?:CORRECT|SUPPORTED|REASON)\s*:|\n|$)", flat, re.S | re.I)
        return " ".join(m.group(1).split()).rstrip(".") if m else ""
    correct = pick("CORRECT").upper()
    supported = pick("SUPPORTED").upper()
    # Only the allowed values count. Anything else is marked UNPARSED and is left out of the summary counts.
    return {"correct": correct if correct in ("CORRECT", "PARTLY CORRECT", "WRONG") else "UNPARSED",
            "supported": supported if supported in ("YES", "NO") else "UNPARSED",
            "reason": pick("REASON")}


# The verdicts, one per query. The query cells below fill this in.
verdicts = {}
print("parse_verdict() defined. The judge cells below fill verdicts one query at a time.")


**PDF step D2.1 (p4):** "run it on all 10 answers, show the results in one table with a row per query (query, correct, supported, reason)." This cell judges Q1.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: judges query Q1 with the judge prompt, and reads the verdict.
# Why: one cell per query, so each verdict and its reason can be read on its own.
# Output: the query, CORRECT, SUPPORTED and REASON.
# ---------------------------------------------------------------------------

QID = "Q1"
# Find this query in QUERIES by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# The 3 reranked chunks the generator saw, in order. The judge checks support against them.
texts = [c["text"] for c in final_chunks[QID]]
# Fill the judge prompt with the question, the expected answer, the 3 chunks and the model's answer.
prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                             chunk_2=texts[1], chunk_3=texts[2], answer=answers[QID])
# Greedy judge reply (temperature 0), so the same answer always gets the same verdict.
raw = judge_generate(prompt)
# Read the three labelled lines. A line outside the format is marked UNPARSED.
verdicts[QID] = dict(parse_verdict(raw), raw=raw)
v = verdicts[QID]
print(f"{QID}  {q['query']}")
print(f"  CORRECT={v['correct']}  SUPPORTED={v['supported']}  REASON={v['reason']}")


**PDF step D2.1 (p4):** "run it on all 10 answers, show the results in one table with a row per query (query, correct, supported, reason)." This cell judges Q2.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: judges query Q2 with the judge prompt, and reads the verdict.
# Why: one cell per query, so each verdict and its reason can be read on its own.
# Output: the query, CORRECT, SUPPORTED and REASON.
# ---------------------------------------------------------------------------

QID = "Q2"
# Find this query in QUERIES by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# The 3 reranked chunks the generator saw, in order. The judge checks support against them.
texts = [c["text"] for c in final_chunks[QID]]
# Fill the judge prompt with the question, the expected answer, the 3 chunks and the model's answer.
prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                             chunk_2=texts[1], chunk_3=texts[2], answer=answers[QID])
# Greedy judge reply (temperature 0), so the same answer always gets the same verdict.
raw = judge_generate(prompt)
# Read the three labelled lines. A line outside the format is marked UNPARSED.
verdicts[QID] = dict(parse_verdict(raw), raw=raw)
v = verdicts[QID]
print(f"{QID}  {q['query']}")
print(f"  CORRECT={v['correct']}  SUPPORTED={v['supported']}  REASON={v['reason']}")


**PDF step D2.1 (p4):** "run it on all 10 answers, show the results in one table with a row per query (query, correct, supported, reason)." This cell judges Q3.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: judges query Q3 with the judge prompt, and reads the verdict.
# Why: one cell per query, so each verdict and its reason can be read on its own.
# Output: the query, CORRECT, SUPPORTED and REASON.
# ---------------------------------------------------------------------------

QID = "Q3"
# Find this query in QUERIES by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# The 3 reranked chunks the generator saw, in order. The judge checks support against them.
texts = [c["text"] for c in final_chunks[QID]]
# Fill the judge prompt with the question, the expected answer, the 3 chunks and the model's answer.
prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                             chunk_2=texts[1], chunk_3=texts[2], answer=answers[QID])
# Greedy judge reply (temperature 0), so the same answer always gets the same verdict.
raw = judge_generate(prompt)
# Read the three labelled lines. A line outside the format is marked UNPARSED.
verdicts[QID] = dict(parse_verdict(raw), raw=raw)
v = verdicts[QID]
print(f"{QID}  {q['query']}")
print(f"  CORRECT={v['correct']}  SUPPORTED={v['supported']}  REASON={v['reason']}")


**PDF step D2.1 (p4):** "run it on all 10 answers, show the results in one table with a row per query (query, correct, supported, reason)." This cell judges Q4.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: judges query Q4 with the judge prompt, and reads the verdict.
# Why: one cell per query, so each verdict and its reason can be read on its own.
# Output: the query, CORRECT, SUPPORTED and REASON.
# ---------------------------------------------------------------------------

QID = "Q4"
# Find this query in QUERIES by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# The 3 reranked chunks the generator saw, in order. The judge checks support against them.
texts = [c["text"] for c in final_chunks[QID]]
# Fill the judge prompt with the question, the expected answer, the 3 chunks and the model's answer.
prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                             chunk_2=texts[1], chunk_3=texts[2], answer=answers[QID])
# Greedy judge reply (temperature 0), so the same answer always gets the same verdict.
raw = judge_generate(prompt)
# Read the three labelled lines. A line outside the format is marked UNPARSED.
verdicts[QID] = dict(parse_verdict(raw), raw=raw)
v = verdicts[QID]
print(f"{QID}  {q['query']}")
print(f"  CORRECT={v['correct']}  SUPPORTED={v['supported']}  REASON={v['reason']}")


**PDF step D2.1 (p4):** "run it on all 10 answers, show the results in one table with a row per query (query, correct, supported, reason)." This cell judges Q5.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: judges query Q5 with the judge prompt, and reads the verdict.
# Why: one cell per query, so each verdict and its reason can be read on its own.
# Output: the query, CORRECT, SUPPORTED and REASON.
# ---------------------------------------------------------------------------

QID = "Q5"
# Find this query in QUERIES by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# The 3 reranked chunks the generator saw, in order. The judge checks support against them.
texts = [c["text"] for c in final_chunks[QID]]
# Fill the judge prompt with the question, the expected answer, the 3 chunks and the model's answer.
prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                             chunk_2=texts[1], chunk_3=texts[2], answer=answers[QID])
# Greedy judge reply (temperature 0), so the same answer always gets the same verdict.
raw = judge_generate(prompt)
# Read the three labelled lines. A line outside the format is marked UNPARSED.
verdicts[QID] = dict(parse_verdict(raw), raw=raw)
v = verdicts[QID]
print(f"{QID}  {q['query']}")
print(f"  CORRECT={v['correct']}  SUPPORTED={v['supported']}  REASON={v['reason']}")


**PDF step D2.1 (p4):** "run it on all 10 answers, show the results in one table with a row per query (query, correct, supported, reason)." This cell judges Q6.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: judges query Q6 with the judge prompt, and reads the verdict.
# Why: one cell per query, so each verdict and its reason can be read on its own.
# Output: the query, CORRECT, SUPPORTED and REASON.
# ---------------------------------------------------------------------------

QID = "Q6"
# Find this query in QUERIES by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# The 3 reranked chunks the generator saw, in order. The judge checks support against them.
texts = [c["text"] for c in final_chunks[QID]]
# Fill the judge prompt with the question, the expected answer, the 3 chunks and the model's answer.
prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                             chunk_2=texts[1], chunk_3=texts[2], answer=answers[QID])
# Greedy judge reply (temperature 0), so the same answer always gets the same verdict.
raw = judge_generate(prompt)
# Read the three labelled lines. A line outside the format is marked UNPARSED.
verdicts[QID] = dict(parse_verdict(raw), raw=raw)
v = verdicts[QID]
print(f"{QID}  {q['query']}")
print(f"  CORRECT={v['correct']}  SUPPORTED={v['supported']}  REASON={v['reason']}")


**PDF step D2.1 (p4):** "run it on all 10 answers, show the results in one table with a row per query (query, correct, supported, reason)." This cell judges Q7.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: judges query Q7 with the judge prompt, and reads the verdict.
# Why: one cell per query, so each verdict and its reason can be read on its own.
# Output: the query, CORRECT, SUPPORTED and REASON.
# ---------------------------------------------------------------------------

QID = "Q7"
# Find this query in QUERIES by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# The 3 reranked chunks the generator saw, in order. The judge checks support against them.
texts = [c["text"] for c in final_chunks[QID]]
# Fill the judge prompt with the question, the expected answer, the 3 chunks and the model's answer.
prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                             chunk_2=texts[1], chunk_3=texts[2], answer=answers[QID])
# Greedy judge reply (temperature 0), so the same answer always gets the same verdict.
raw = judge_generate(prompt)
# Read the three labelled lines. A line outside the format is marked UNPARSED.
verdicts[QID] = dict(parse_verdict(raw), raw=raw)
v = verdicts[QID]
print(f"{QID}  {q['query']}")
print(f"  CORRECT={v['correct']}  SUPPORTED={v['supported']}  REASON={v['reason']}")


**PDF step D2.1 (p4):** "run it on all 10 answers, show the results in one table with a row per query (query, correct, supported, reason)." This cell judges Q8.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: judges query Q8 with the judge prompt, and reads the verdict.
# Why: one cell per query, so each verdict and its reason can be read on its own.
# Output: the query, CORRECT, SUPPORTED and REASON.
# ---------------------------------------------------------------------------

QID = "Q8"
# Find this query in QUERIES by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# The 3 reranked chunks the generator saw, in order. The judge checks support against them.
texts = [c["text"] for c in final_chunks[QID]]
# Fill the judge prompt with the question, the expected answer, the 3 chunks and the model's answer.
prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                             chunk_2=texts[1], chunk_3=texts[2], answer=answers[QID])
# Greedy judge reply (temperature 0), so the same answer always gets the same verdict.
raw = judge_generate(prompt)
# Read the three labelled lines. A line outside the format is marked UNPARSED.
verdicts[QID] = dict(parse_verdict(raw), raw=raw)
v = verdicts[QID]
print(f"{QID}  {q['query']}")
print(f"  CORRECT={v['correct']}  SUPPORTED={v['supported']}  REASON={v['reason']}")


**PDF step D2.1 (p4):** "run it on all 10 answers, show the results in one table with a row per query (query, correct, supported, reason)." This cell judges Q9.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: judges query Q9 with the judge prompt, and reads the verdict.
# Why: one cell per query, so each verdict and its reason can be read on its own.
# Output: the query, CORRECT, SUPPORTED and REASON.
# ---------------------------------------------------------------------------

QID = "Q9"
# Find this query in QUERIES by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# The 3 reranked chunks the generator saw, in order. The judge checks support against them.
texts = [c["text"] for c in final_chunks[QID]]
# Fill the judge prompt with the question, the expected answer, the 3 chunks and the model's answer.
prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                             chunk_2=texts[1], chunk_3=texts[2], answer=answers[QID])
# Greedy judge reply (temperature 0), so the same answer always gets the same verdict.
raw = judge_generate(prompt)
# Read the three labelled lines. A line outside the format is marked UNPARSED.
verdicts[QID] = dict(parse_verdict(raw), raw=raw)
v = verdicts[QID]
print(f"{QID}  {q['query']}")
print(f"  CORRECT={v['correct']}  SUPPORTED={v['supported']}  REASON={v['reason']}")


**PDF step D2.1 (p4):** "run it on all 10 answers, show the results in one table with a row per query (query, correct, supported, reason)." This cell judges Q10.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: judges query Q10 with the judge prompt, and reads the verdict.
# Why: one cell per query, so each verdict and its reason can be read on its own.
# Output: the query, CORRECT, SUPPORTED and REASON.
# ---------------------------------------------------------------------------

QID = "Q10"
# Find this query in QUERIES by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# The 3 reranked chunks the generator saw, in order. The judge checks support against them.
texts = [c["text"] for c in final_chunks[QID]]
# Fill the judge prompt with the question, the expected answer, the 3 chunks and the model's answer.
prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                             chunk_2=texts[1], chunk_3=texts[2], answer=answers[QID])
# Greedy judge reply (temperature 0), so the same answer always gets the same verdict.
raw = judge_generate(prompt)
# Read the three labelled lines. A line outside the format is marked UNPARSED.
verdicts[QID] = dict(parse_verdict(raw), raw=raw)
v = verdicts[QID]
print(f"{QID}  {q['query']}")
print(f"  CORRECT={v['correct']}  SUPPORTED={v['supported']}  REASON={v['reason']}")


**PDF step D2.1 (p4):** "show the results in one table with a row per query (query, correct, supported, reason), and fill the summary below. Count out of 10."

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: prints the D2.1 results table for all 10 queries, and the summary counts.
# Why: the brief asks for one table with a row per query, and a summary out of 10.
# Output: the table, the summary counts, and any reply the judge did not follow the format for.
# ---------------------------------------------------------------------------

query_text = {q["qid"]: q["query"] for q in QUERIES}
# A verdict with UNPARSED in either check is left out of the counts.
unparsed = [qid for qid, v in verdicts.items() if "UNPARSED" in (v["correct"], v["supported"])]
show_md_table(["Query", "Correct", "Supported", "Reason"],
              [(f"{qid}: {query_text[qid]}", v["correct"], v["supported"], v["reason"]) for qid, v in verdicts.items()],
              title="D2.1 judge results", save_as="d2_verdicts")
# Count the queries that got each correctness label.
counts = {k: sum(v["correct"] == k for v in verdicts.values()) for k in ("CORRECT", "PARTLY CORRECT", "WRONG")}
print_table(["Check", "Count out of 10"], [
    ("Correct", str(counts["CORRECT"])),
    ("Partly correct", str(counts["PARTLY CORRECT"])),
    ("Wrong", str(counts["WRONG"])),
    ("Supported by the chunks", str(sum(v["supported"] == "YES" for v in verdicts.values()))),
], title="D2.1 summary", save_as="d2_summary")
# Print the replies that did not follow the format, so they can be read.
print(f"lines the judge did not follow the format: {len(unparsed)} {unparsed}")
for qid in unparsed:
    print(f"  {qid} judge reply: {' '.join(verdicts[qid]['raw'].split())[:300]}")


#### D2.2 Spot-check the judge (1 mark)

*The cell below shows 3 answers (with different verdicts where possible), their 3 chunks and the judge's verdicts. Read them, then fill in the notes cell after it.*

**PDF step D2.2 (p4):** "Spot-check the judge: read 3 of the answers and their chunks yourself. Say how many of the judge's verdicts you agree with, and explain one disagreement if there is one."

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: shows the first 3 answers with their chunks and the judge's verdicts, so you can read them yourself.
# Why: the brief asks you to read 3 answers and their chunks, then say how many verdicts you agree with.
# Output: the 3 answers, their chunks, and the judge's verdicts.
# ---------------------------------------------------------------------------

for q in QUERIES[:3]:
    qid = q["qid"]
    print(f"\n{qid}  {q['query']}")
    print(f"  Expected answer: {q['expected_answer']}")
    print(f"  Model answer: {answers[qid]}")
    for i, c in enumerate(final_chunks[qid], 1):
        print(f"  [{i}] {c['text']}")
    print(f"  Judge: CORRECT={verdicts[qid]['correct']}  SUPPORTED={verdicts[qid]['supported']}  REASON={verdicts[qid]['reason']}")


**D2.2 my spot-check** *(fill in after reading the 3 answers above)*

| Query | Judge (correct / supported) | Mine (correct / supported) | Agree? |
| --- | --- | --- | --- |
| Q_ | | | |
| Q_ | | | |
| Q_ | | | |

I agree with _ of the 3 verdicts. Disagreement (if any): _one or two sentences on which verdict and why._

#### D2.3 Find the cause (1 mark)

*The cell below picks one answer marked WRONG or SUPPORTED: NO, shows its 3 chunks, and checks whether the answer span is in them: was the needed information in the chunks or not?*


**PDF step D2.3 (p5):** "Find the cause: pick one answer that is wrong or not supported, and show its 3 chunks. Say whether retrieval or generation caused the problem: was the needed information in the chunks or not?"

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: picks the first answer marked WRONG or SUPPORTED: NO, shows its 3 chunks, and checks the answer span.
# Why: the brief asks for one such answer, its 3 chunks, and whether the needed information was in those chunks.
# Output: the answer, its chunks, and whether the answer span is in them.
# ---------------------------------------------------------------------------


def failed(qid):
    """True if the judge marked the answer WRONG on correctness, or NO on support."""
    return verdicts[qid]["correct"] == "WRONG" or verdicts[qid]["supported"] == "NO"


# The first query, in order, that failed. None if every query passed.
FAILED_QID = next((q["qid"] for q in QUERIES if failed(q["qid"])), None)
if FAILED_QID is None:
    print("No answer is WRONG or unsupported, so there is nothing to analyse.")
else:
    q = next(x for x in QUERIES if x["qid"] == FAILED_QID)
    print(f"{FAILED_QID}  {q['query']}")
    print(f"  Expected answer: {q['expected_answer']}")
    print(f"  Model answer: {answers[FAILED_QID]}")
    print(f"  Judge: CORRECT={verdicts[FAILED_QID]['correct']}  SUPPORTED={verdicts[FAILED_QID]['supported']}  "
          f"REASON={verdicts[FAILED_QID]['reason']}")
    for i, c in enumerate(final_chunks[FAILED_QID], 1):
        print(f"\n  [{i}] {c['chunk_id']}:\n{c['text']}")
    # Was the needed information in the 3 chunks? The answer span is that information.
    in_chunks = any(is_hit(c["text"], q["span"]) for c in final_chunks[FAILED_QID])
    print(f"\n  Answer span in the 3 chunks: {'yes' if in_chunks else 'no'}")


**D2.3 cause in my words** *(fill in after reading the 3 chunks above)*: _was the needed information in the chunks or not, and what did the model do with it? One or two sentences._

#### D2.4 Suggest a fix (1 mark)

*One change that matches the cause found in D2.3, and why it would help.*